<h1 style="font-size: 35pt; color: #58B2DC; font-family: sans-serif; font-weight: bold;">Introduction</h1>

<div style="font-size: 15pt; font-family: sans-serif;">
    Welcome to my notebook!<br>
    In this notebook, I will introduce <b>Pytorch</b>. Pytorch is one of Neural Network libraries. Other famous library is Tensorflow, maybe you here the name. Tensorflow is easy to build models, comparing with Pytorch. But Pytorch is easier than Tensorflow to extend model, customize layers and change something. <br>
    In Kaggle Competition, you will try various models, change your model, experiment what is the best. Pytorch will help you and be powerful tool for you.
</div>

<h1 style="font-size: 35pt; color: #58B2DC; font-family: sans-serif; font-weight: bold;">Import Libraries</h1>

<div style="font-size: 15pt; font-family: sans-serif;">
    At first I install libraries that I will be using. I append some explanation.
</div>

In [ ]:
# tor getting time
import datetime
# for mathmatical calculation
import numpy as np 
# to treat tabular data
import pandas as pd
# to transform data
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

# pytorch
import torch
# pytorch optimizer: train and update weight by this modules
import torch.optim as optim
# pytorch layers: build models by this modules
import torch.nn as nn
# install some functions
import torch.nn.functional as F
# to treat data easier
from torch.utils.data import random_split, TensorDataset, DataLoader

# to plot graphs
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
plt.style.use('fivethirtyeight') # set style, for more information check Reference section and click "Matplotlib Style"

# to reproduce same result for each running notebook
SEED = 42

<h1 style="font-size: 35pt; color: #58B2DC; font-family: sans-serif; font-weight: bold;">Read Data and Feature Engineerning</h1>

<div style="font-size: 15pt; font-family: sans-serif;">
    Next, I read train data and transform. For feature engineering, transforming data, I explain a bit in the notebook: <a ref="https://www.kaggle.com/code/masatakasuzuki/titanic-your-first-notebook-for-tabular-data" style="border-bottom:solid; border-color: #808080; border-width: 2px;">🚢Titanic - Your First Notebook for Tabular Data</a>. Please check it!
</div>

<div style="font-size: 15pt; font-family: sans-serif;">
    Read data by pandas.read_csv.
</div>

In [ ]:
df_train = pd.read_csv('/kaggle/input/titanic/train.csv')

<div style="font-size: 15pt; font-family: sans-serif;">
    Define feature engineering function.
</div>

In [ ]:
def titanic_preprocessing(input_df):
    df = input_df.copy()
    # missin values
    # Age is int(or float) so use mean or median
    # Embarked is caterogy(label) so use mode
    df['Age'] = df['Age'].fillna(df['Age'].median())
    df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
    df['Fare'] = df['Fare'].fillna(df['Fare'].median()) # Fare is missing in test data
    # extract title, we can know gender, work etc.
    df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
    df['Title'] = df['Title'].replace(['Lady', 'Countess','Capt', 'Col','Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
    df['Title'] = df['Title'].replace('Mlle', 'Miss')
    df['Title'] = df['Title'].replace('Ms', 'Miss')
    df['Title'] = df['Title'].replace('Mme', 'Mrs')
    # Label Encoding
    # let the string category feature to integer
    le = LabelEncoder()
    df['Sex'] = le.fit_transform(df['Sex'])
    df['Embarked'] = le.fit_transform(df['Embarked'])
    df['Title'] = le.fit_transform(df['Title'])
    # Standardize
    # Scaling int or float feature, it improve prediction for especially Neural Network Model.
    sc = StandardScaler()
    df[['Age', 'Fare']] = sc.fit_transform(df[['Age', 'Fare']])
    # Get the number of passenger in each group
    df['PassengersInGroup'] = df['SibSp'] + df['Parch'] + 1 # Siblings/Spouses + Parent/Children + him/herself
    df['IsAlone'] = df['PassengersInGroup'].apply(lambda x: 1 if x == 1 else 0)
    # drop sibsp and parch because these have high correlation for PassengersInGroup
    df = df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin', 'SibSp', 'Parch'])
    return df

<div style="font-size: 15pt; font-family: sans-serif;">
    Apply feature engineering function for train data.
</div>

In [ ]:
df_train = titanic_preprocessing(df_train)
df_train.head()

<div style="font-size: 15pt; font-family: sans-serif;">
    We can see that string values are changed to integer. Also Age and Fare are standardized and some columns are added and removed.<br>
    We define the columns to send as train data: feature_cols, and as label column: target_col.
</div>

In [ ]:
feature_cols = df_train.columns.tolist()
target_col = 'Survived'
feature_cols.remove(target_col)

<h1 style="font-size: 35pt; color: #58B2DC; font-family: sans-serif; font-weight: bold;">Build and Train your Model</h1>

<div style="font-size: 15pt; font-family: sans-serif;">
    Next, Let's build your model and train data! Also check the result by ploting graphs.<br>
    I will explain step by step. So let's dive in!
</div>

<div style="font-size: 15pt; font-family: sans-serif;">
    I define simple three layers Neural Network Model as below.
</div>

In [ ]:
class NNModel(nn.Module):
    def __init__(self, input_shape):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_shape, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Dropout(0.5)
        )
        self.out_layer = nn.Sequential(
            nn.Linear(8, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        x = self.layers(x)
        y = self.out_layer(x)
        return y

<div style="font-size: 15pt; font-family: sans-serif;">
    Let's check details.<br>
    <b>nn.Module</b> is pytorch class to build neural network model. It calculate weight updating, predictiction and so forth. To use nn.Module functions, we need to call <b>super().__init__()</b> in init function fo our class we define above.<br>
    "self.layers" and "self.out_layer" is the layers definition we use. If you don't know the mean of each layer, <a ref="https://www.kaggle.com/learn/intro-to-deep-learning" style="border-bottom:solid; border-color: #808080; border-width: 2px;">Kaggle Lean "Intro to Deep Learning"</a> is healpful.<br>
    <b>forward</b> is the model definition. In this model, Input "x" is transformed by "self.layers" and "self.out_layer". It is easy to change if you want to change layers. You can change self.layers or processes in forward function. You can add layers, devide outputs and inputs, and so on. 
</div>

<div style="font-size: 15pt; font-family: sans-serif;">
    Next, train model by training data.<br>
    At first, to get same result(reproducibility), I set random seed.
</div>

In [ ]:
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.manual_seed(SEED)
np.random.seed(SEED)

<div style="font-size: 15pt; font-family: sans-serif;">
    Choose training data and validation data. Training data is used to train model. Validation data is used to check how good the model we train. To devide data to training and validation, <b>train_test_split</b> is useful.
</div>

In [ ]:
X, y = df_train[feature_cols].values, df_train[[target_col]].values
x_train, x_val, y_train, y_val = train_test_split(X, y, test_size=0.2, shuffle=True, stratify=y, random_state=SEED)

<div style="font-size: 15pt; font-family: sans-serif;">
    To update weights in our model, we need pytorch tensor. Let's transform train and validation data. to change data to tensor, we use <b>torch.as_tensor</b>.<br>
    When we train data, mini batch is useful to avoid overfitting or trapped in local maximum. To use mini batch, <b>TensorDataset</b> and <b>DataLoader</b> is useful.
</div>

In [ ]:
x_train_tensor = torch.as_tensor(x_train, dtype=torch.float)
y_train_tensor = torch.as_tensor(y_train, dtype=torch.float)
x_val_tensor = torch.as_tensor(x_val, dtype=torch.float)
y_val_tensor = torch.as_tensor(y_val, dtype=torch.float)
train_dataset = TensorDataset(x_train_tensor, y_train_tensor)
val_dataset = TensorDataset(x_val_tensor, y_val_tensor)
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=32,
    shuffle=True,
)
val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=32,
)

<div style="font-size: 15pt; font-family: sans-serif;">
    if you want to use GPU in pytorch. we should define device, if we can use GPU <b>torch.cuda.is_available</b> is True.
</div>

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

<div style="font-size: 15pt; font-family: sans-serif;">
    Next, we define model, loss_fn, optimizer. <br>
    loss_fn is the loss function. we want to minimize the output of this function. For 0 or 1 classification problem, BCELoss(Binary Cross Entropy Loss) is good.<br>
    optimzier is used for updating weight in our model. In this notebook, I choose Adam.<br>
    <b>to(device)</b> method is used to use GPU.
</div>

In [ ]:
# define model. don't forget to(device) if you use GPU
model = NNModel(x_train_tensor.size()[1]).to(device)
# loss function
loss_fn = nn.BCELoss()
# optimizer
optimizer = optim.Adam(model.parameters(), lr=0.01)

<div style="font-size: 15pt; font-family: sans-serif;">
    Next, finally, we train model.
    <ol>
        <li>
            I define loop. In this notebook I use 100 loop to improve model. Weights is changed a bit in each loop.
        </li>
        <li>
            In each loop, I use mini batch. mini batch is extracted from data loader.
        </li>
        <li>
            we change model to training mode by calling <b>model.tarin</b>.
        </li>
        <li>
            To update weight, gradient is used in pytorch library. I reset gradient to calculate losses.
        </li>
        <li>
            At <b>model(x_batch)</b>, pytorch library call "forward" function we build in NNModel. we get prediction here.
        </li>
        <li>
            To use loss_fn, we can get loss. 
        </li>
        <li>
            <b>loss.backward()</b> is used to calculate gradient.
        </li>
        <li>
            We can update weights in our model by <b>optimizer.step()</b>.
        </li>
        <li>
            <b>loss.item()</b> is used to transform loss from tensor type to numpy type.
        </li>
    </ol>
    After training, it is helpful to evaluate model to know how godd the model become.
    <ol>
        <li>
            To save cpu and gpu resources, I call <b>with torch.no_grad()</b> not to calculate gradient.
        </li>
        <li>
            <b>model.eval()</b> is used to change model to validation mode.
        </li>
</div>

In [ ]:
n_epochs = 100
losses = []
val_losses = []
for epoch in range(1, n_epochs+1):
    mini_batch_losses = []
    for x_batch, y_batch in train_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)

        model.train()
        # reset gradient
        optimizer.zero_grad()
        # predict from train data
        yhat = model(x_batch)
        # compute loss
        loss = loss_fn(yhat, y_batch)
        # calculate gradient
        loss.backward()
        # update weight parameters
        optimizer.step()
        mini_batch_loss = loss.item()
        mini_batch_losses.append(mini_batch_loss)

    loss = np.mean(mini_batch_losses)
    losses.append(loss)

    # no gradient in validation mode
    with torch.no_grad():
        mini_batch_losses = []
        for x_batch, y_batch in val_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)

            # set model to validation mode
            model.eval()
            # predict from validation data
            yhat = model(x_batch)
            # compute loss
            loss = loss_fn(yhat, y_batch)
            mini_batch_loss = loss.item()
            mini_batch_losses.append(mini_batch_loss)
        val_loss = np.mean(mini_batch_losses)
        val_losses.append(val_loss)

<div style="font-size: 15pt; font-family: sans-serif;">
    Let's vidualize training process.
</div>

In [ ]:
fig = plt.figure(figsize=(10, 6))
plt.plot(losses, label='Trainig Loss', c='#0000ff', linestyle='solid')
plt.plot(val_losses, label='Validation Loss', c='#87cefa', linestyle='dashed')
plt.yscale('log')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.tight_layout()
plt.show()

<div style="font-size: 15pt; font-family: sans-serif;">
    Blue solid line is the value of training loss and light blue dotted line is the value of validation loss.<br>
    As you can see, two loss is decreasing to 40 epoch and almost flat from 40 to 100 epoch.
</div>

<h1 style="font-size: 35pt; color: #58B2DC; font-family: sans-serif; font-weight: bold;">Advanced Step - Build and Train by using Python Class</h1>

<div style="font-size: 15pt; font-family: sans-serif;">
    I introduce a bit different and elegant way to build and train your model. Creating Python class and function, we can build and train models simply like <a ref="https://scikit-learn.org/stable/" style="border-bottom:solid; border-color: #808080; border-width: 2px;">Sklearn</a>. <br>
    This will help you to apply <b>Ensemble</b> to your Neural Network model. I introduced some ensemble methods in the notebook: <a ref="https://www.kaggle.com/code/masatakasuzuki/titanic-4-ensemble-methods" style="border-bottom:solid; border-color: #808080; border-width: 2px;">🚢Titanic - 4 Ensemble Methods</a>. Ensemble helps to reduce noises and improve your prediction.
</div>

<div style="font-size: 15pt; font-family: sans-serif;">
    I use same neural network model above.
</div>

In [ ]:
class NNModel(nn.Module):
    def __init__(self, input_shape):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_shape, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(8, 1),
            nn.Sigmoid(),
        )
    
    def forward(self, x):
        y = self.layers(x)
        return y

<div style="font-size: 15pt; font-family: sans-serif;">
    The difference is to create training and prediction mechanism in the class below.<br>
    In the class, doing same thing above.
</div>

In [ ]:
class NNTrainer():
    def __init__(self, model, loss_fn, optimizer):
        # model
        self.model = model
        # loss function
        self.loss_fn = loss_fn
        # optimizer
        self.optimizer = optimizer
        # to use cuda, set device
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.model.to(self.device)
        
        # loader(train and validation data)
        self.train_loader = None
        self.val_loader = None
        
        # variables for training history
        self.losses = []
        self.val_losses = []
        self.total_epochs = 0
        
    def set_loader(self, train_loader, val_loader=None):
        # set train and validation loader
        self.train_loader = train_loader
        self.val_loader = val_loader
        
    def set_seed(self, seed=42):
        # to reproducibility, set seed to pytorch and numpy
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        torch.manual_seed(seed)
        np.random.seed(seed)
        
    def _make_train_step_fn(self):
        def perform_train_step_fn(x, y):
            # set model to train mode
            self.model.train()
            # reset gradient
            optimizer.zero_grad()
            # predict from train data
            yhat = self.model(x)
            # compute loss
            loss = self.loss_fn(yhat, y)
            # calculate gradient
            loss.backward()
            # update weight parameters
            optimizer.step()
            return loss.item() # to numpy values from tensor, use item method
        return perform_train_step_fn
    
    def _make_val_step_fn(self):
        def perform_val_step_fn(x, y):
            # set model to validation mode
            self.model.eval()
            # predict from validation data
            yhat = self.model(x)
            # compute loss
            loss = self.loss_fn(yhat, y)
            return loss.item() # to numpy values from tensor, use item method
        return perform_val_step_fn
    
    def _mini_batch(self, validation=False):
        if validation:
            data_loader = self.val_loader
            step_fn = self._make_val_step_fn()
        else:
            data_loader = self.train_loader
            step_fn = self._make_train_step_fn()
            
        if data_loader is None:
            return None
        
        mini_batch_losses = []
        for x_batch, y_batch in data_loader:
            x_batch, y_batch = x_batch.to(self.device), y_batch.to(self.device)
            
            mini_batch_loss = step_fn(x_batch, y_batch)
            mini_batch_losses.append(mini_batch_loss)
            
        loss = np.mean(mini_batch_losses)
        return loss
            
    def fit(self, n_epochs, train_loader, val_loader=None, random_state=42):
        # to ensure reproducibility
        self.set_seed(random_state)
        # set loader
        self.set_loader(train_loader, val_loader)
        
        for epoch in range(1, n_epochs+1):
            self.total_epochs += 1
            
            loss = self._mini_batch(validation=False)
            self.losses.append(loss)
            
            # no gradient in validation mode
            with torch.no_grad():
                val_loss = self._mini_batch(validation=True)
                self.val_losses.append(val_loss)
                
    def save_checkpoint(self, filename):
        # if you resume training and continue other day, save checkpoint
        checkpoint = {
            'epoch': self.total_epochs,
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'loss': self.losses,
            'val_loss': self.val_losses,
        }
        torch.save(checkpoint, filename)
        
    def load_checkpoint(self, filename):
        checkpoint = torch.load(filename)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        self.total_epochs = checkpoint['epoch']
        self.losses = checkpoint['loss']
        self.val_losses = checkpoint['val_loss']
        
    def predict_proba(self, x):
        self.model.eval()
        x_tensor = torch.as_tensor(x).float()
        y_hat_tensor = self.model(x_tensor.to(self.device))
        return y_hat_tensor.reshape(-1).detach().cpu().numpy()
    
    def predict(self, x):
        self.model.eval()
        x_tensor = torch.as_tensor(x).float()
        y_hat_tensor = self.model(x_tensor.to(self.device))
        y_hat = y_hat_tensor.detach().cpu().numpy()
        y_hat = np.array([1 if x >= 0.5 else 0 for x in y_hat])
        return y_hat
    
    def plot_losses(self):
        fig = plt.figure(figsize=(10, 6))
        plt.plot(self.losses, label='Trainig Loss', c='#0000ff', linestyle='solid')
        if self.val_loader:
            plt.plot(self.val_losses, label='Validation Loss', c='#87cefa', linestyle='dashed')
        plt.yscale('log')
        plt.xlabel('Epochs')
        plt.ylabel('Loss')
        plt.legend()
        plt.tight_layout()
        plt.show()

<div style="font-size: 15pt; font-family: sans-serif;">
    What I will do is same as "Build and Train Your Model" section. the one thing I added is checkpoint. to reuse the model, We can save weights and results by save_checkpoint and can load weights and results by load_checkpoint.
</div>

<div style="font-size: 15pt; font-family: sans-serif;">
    Below cell is same as the section we see before.
</div>

In [ ]:
X, y = df_train[feature_cols].values, df_train[[target_col]].values
x_train, x_val, y_train, y_val = train_test_split(X, y, test_size=0.2, shuffle=True, stratify=y, random_state=SEED)

In [ ]:
torch.manual_seed(SEED)

x_train_tensor = torch.as_tensor(x_train, dtype=torch.float)
y_train_tensor = torch.as_tensor(y_train, dtype=torch.float)
x_val_tensor = torch.as_tensor(x_val, dtype=torch.float)
y_val_tensor = torch.as_tensor(y_val, dtype=torch.float)
train_dataset = TensorDataset(x_train_tensor, y_train_tensor)
val_dataset = TensorDataset(x_val_tensor, y_val_tensor)
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=32,
    shuffle=True,
)
val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=32,
)

# define model.
model = NNModel(x_train_tensor.size()[1])
# loss function
loss_fn = nn.BCELoss()
# optimizer
optimizer = optim.Adam(model.parameters(), lr=0.01)

<div style="font-size: 15pt; font-family: sans-serif;">
    As you can see below, by NNTrainer, we can train model only using "fit" method. That's all!
</div>

In [ ]:
model_nn = NNTrainer(model, loss_fn, optimizer)
model_nn.fit(n_epochs=100, train_loader=train_loader, val_loader=val_loader, random_state=SEED)
model_nn.plot_losses()
model_nn.predict_proba(x_val_tensor)

<h1 style="font-size: 35pt; color: #58B2DC; font-family: sans-serif; font-weight: bold;">
    Reference
</h1>

<div style="font-size: 15pt; font-family: sans-serif;">
    <ul>
        <li>
            <a ref="https://pytorch.org/docs/stable/index.html" style="border-bottom:solid; border-color: #808080; border-width: 2px;">Pytorch</a>
        </li>
        <li>
            <a ref="https://www.tensorflow.org/api_docs/python/tf" style="border-bottom:solid; border-color: #808080; border-width: 2px;">Tensorflow</a>
        </li>
        <li>
            <a ref="https://matplotlib.org/stable/gallery/style_sheets/fivethirtyeight.html" style="border-bottom:solid; border-color: #808080; border-width: 2px;">Matplotlib Style</a>
        </li>
    </ul>
</div>
